<a href="https://colab.research.google.com/github/fatoufall737/atelier-scikit-learn-iot/blob/main/atelier_scikit_learn_iot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import files
uploaded = files.upload()  # sélectionne mesures_capteurs.csv depuis mon ordinateur

Saving mesures_capteurs.csv to mesures_capteurs.csv


In [3]:
df = pd.read_csv("mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [4]:
#explorer le dataframe
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


Partie 1 – Gestion des doublons
Avec Pandas,
1) vérifier l’existence de doublons dans df
2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [5]:
#Question 1 : vérifie l'existence de doublons dans df.
df.duplicated().sum()

np.int64(5)

In [6]:
#Question 2 : supprime les doublons puis vérifie la suppression.
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

Partie 2 – Sélection de y (cible) et X (caractéristiques)
1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite",
"pression" et "consommation" comme caractéristiques ou variables explicatives
2) Afficher les cinq premières lignes de X et de y
3) Quel est le type du problème de machine learning ?

In [7]:
#Question 1 : définis "etat" comme la cible (y) et "temperature", "humidite", "pression", "consommation" comme caractéristiques (X).
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

In [9]:
#Question 2 : affiche les cinq premières lignes de X et de y.
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [11]:
y.head()

,etat
0,OK
1,OK
2,OK
3,OK
4,OK


**3) Type du problème de Machine Learning**

C'est un problème de **classification** : la cible `y` (l'état du capteur) est une variable catégorielle avec un nombre limité de valeurs possibles (OK, ALERTE, ERREUR), pas une valeur numérique continue. On cherche à prédire à quelle catégorie appartient chaque mesure, pas à estimer un nombre.

Partie 3 – Découpage Train/Test
Diviser X en deux ensembles distincts : un pour l'entraînement (train) et un pour le test (test).
Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du
découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que
dans les données d'origine.



Consigne : diviser X en train/test, avec 20% pour le test, reproductibilité garantie, et proportions de classes conservées entre train et test.


In [14]:
# On retire les lignes où la cible (etat) est manquante, avant le découpage Train/Test
lignes_valides = y.notna()
X = X[lignes_valides]
y = y[lignes_valides]

print("Nombre de lignes après suppression des y manquants :", X.shape[0])

Nombre de lignes après suppression des y manquants : 596


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(476, 4) (120, 4)


Partie 4 – Gestion des valeurs manquantes
1) Vérifier l’existence de valeurs manquantes
2) Sélectionner SimpleImputer avec la médiane
3) Qu’est ce qui justifie le choix de la médiane ?
4) Trouver les paramètres (médianes) de l’imputeur sur X_train
5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test

In [18]:
X_train.isnull().sum()

,0
temperature,5
humidite,4
pression,5
consommation,3


In [19]:
#Question 2 : sélectionne SimpleImputer avec la médiane.
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

**3) Justification du choix de la médiane**

La médiane est préférée à la moyenne ici car elle est **moins sensible aux valeurs extrêmes (outliers)**. On sait déjà (ateliers précédents) que le jeu de données contient des anomalies très éloignées de la normale (ex. une température à -18°C ou 58°C). Si on utilisait la moyenne, ces valeurs extrêmes fausseraient le calcul et donneraient une valeur de remplacement peu représentative. La médiane reste stable face à ces anomalies, car elle ne dépend que de la position centrale des données triées, pas de leur amplitude.

In [20]:
#Question 4 : trouve les paramètres (médianes) de l'imputeur sur X_train.
imputer.fit(X_train)
imputer.statistics_

array([  24.9 ,   65.38, 1012.3 ,  206.59])

In [22]:
#Question 5 : détermine X_train_imputed et X_test_imputed.
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Valeurs manquantes restantes dans X_train_imputed :", pd.DataFrame(X_train_imputed).isnull().sum().sum())
print("Valeurs manquantes restantes dans X_test_imputed  :", pd.DataFrame(X_test_imputed).isnull().sum().sum())

Valeurs manquantes restantes dans X_train_imputed : 0
Valeurs manquantes restantes dans X_test_imputed  : 0


Partie 5 – Mise à l'échelle
1) Sélectionner StandardScaler pour mettre à l’échelle les transformés de l’imputation
2) Qu’est ce qui justifie la standardisation ?
3) Trouver les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed
4) Déterminer X_train_scaled et X_test_scaled, les transformés de X_train_imputed et
X_test_imputed

In [23]:
#Question 1 : sélectionne StandardScaler pour mettre à l'échelle les transformés de l'imputation.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

**2) Justification de la standardisation**

Les 4 caractéristiques (température, humidité, pression, consommation) n'ont pas la même échelle de valeurs — par exemple, la pression tourne autour de 1000, alors que la température est autour de 20-30. Le modèle KNN (utilisé en Partie 6) calcule des **distances géométriques** entre les points pour faire ses prédictions : sans standardisation, la pression dominerait artificiellement le calcul de distance simplement parce que ses valeurs sont plus grandes, même si elle n'est pas plus importante que les autres variables. La standardisation ramène toutes les colonnes à la même échelle (moyenne 0, écart-type 1), pour que chaque variable ait un poids comparable dans le calcul.

In [24]:
#Question 3 : trouve les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed.
scaler.fit(X_train_imputed)
print("Moyennes :", scaler.mean_)
print("Écarts-types :", scaler.scale_)

Moyennes : [  24.9637395    64.85044118 1012.07621849  210.15394958]
Écarts-types : [ 4.19303979 10.16279173 10.9941836  73.42092322]


In [25]:
#Question 4 : détermine X_train_scaled et X_test_scaled.
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Moyenne après scaling (train) :", X_train_scaled.mean(axis=0).round(2))
print("Écart-type après scaling (train) :", X_train_scaled.std(axis=0).round(2))

Moyenne après scaling (train) : [-0.  0. -0.  0.]
Écart-type après scaling (train) : [1. 1. 1. 1.]
